In [6]:
import sympy as sp
import numpy as np

def matrix_to_latex(matrix, precision):
    """Chuyển ma trận numpy sang định dạng LaTeX pmatrix"""
    if len(matrix.shape) == 1:
        matrix = matrix.reshape(-1, 1)
    lines = []
    for row in matrix:
        lines.append(" & ".join([f"{val:.{precision}f}" for val in row]))
    return "\\begin{pmatrix} " + " \\\\ ".join(lines) + " \\end{pmatrix}"

def newton_modified_detailed():
    print("=== CHƯƠNG TRÌNH GIẢI HỆ PHƯƠNG TRÌNH PHI TUYẾN (NEWTON CẢI TIẾN) ===\n")
    
    try:
        # --- 0. Nhập liệu ---
        raw_f = input("Nhập danh sách các f(x) (vd: f_1(x), f_2(x), f_3(x)): ").split(',')
        variables_str = input("Nhập các biến (ví dụ: x1 x2 x3): ")
        x0_vals = [float(v) for v in input("Khởi tạo X0 nên chọn lại nhiều lần (ví dụ: 1 1 1) (CHÚ Ý: det(J)=0): ").split()]
        n_iterations = int(input("Nhập số bước lặp (n): "))
        precision = int(input("Nhập số chữ số làm tròn (precision): "))

        # --- 1. Thiết lập toán học ---
        vars = sp.symbols(variables_str)
        F_expr = sp.Matrix([sp.sympify(f.strip()) for f in raw_f])
        J_expr = F_expr.jacobian(vars)
        
        f_func = sp.lambdify([vars], F_expr, 'numpy')
        j_func = sp.lambdify([vars], J_expr, 'numpy')

        X0 = np.array(x0_vals, dtype=float).reshape(-1, 1)
        F_at_X0 = f_func(X0.flatten()).astype(float)
        J_at_X0 = j_func(X0.flatten()).astype(float)
        
        try:
            J_inv_fixed = np.linalg.inv(J_at_X0)
        except np.linalg.LinAlgError:
            print("\n[!] LỖI: Ma trận Jacobi tại X0 bị suy biến!")
            return

        # --- 2. Trình bày Bước 1 cực kỳ chi tiết ---
        print("\n" + "="*60)
        print("### 1. Thông tin hệ phương trình và Thiết lập ban đầu")
        print(f"* **Hệ phương trình:** $F(X) = {sp.latex(F_expr.T)}^T$")
        print(f"* **Ma trận Jacobi tổng quát:** $J(X) = {sp.latex(J_expr)}$")
        print(f"* **Giá trị khởi đầu:** $X_0 = {matrix_to_latex(X0, precision)}$")

        print("\n### 2. Chi tiết bước lặp 1 (k = 0)")
        print(f"* **Giá trị hàm số tại $X_0$:**")
        print(f"  $$F(X_0) = {matrix_to_latex(F_at_X0, precision)}$$")
        
        print(f"* **Ma trận Jacobi tại $X_0$:**")
        print(f"  $$J(X_0) = {matrix_to_latex(J_at_X0, precision)}$$")
        
        print(f"* **Ma trận nghịch đảo Jacobi (cố định cho mọi bước):**")
        print(f"  $$J(X_0)^{{-1}} = {matrix_to_latex(J_inv_fixed, precision)}$$")

        # Tính X1
        delta_X0 = J_inv_fixed @ F_at_X0
        X1 = X0 - delta_X0
        
        print(f"* **Tính toán nghiệm xấp xỉ $X_1$:**")
        print(f"  $$X_1 = X_0 - J(X_0)^{{-1}} \\cdot F(X_0)$$")
        print(f"  $$X_1 = {matrix_to_latex(X0, precision)} - {matrix_to_latex(J_inv_fixed, precision)} \\cdot {matrix_to_latex(F_at_X0, precision)}$$")
        print(f"  $$X_1 = {matrix_to_latex(X1, precision)}$$")

        # --- 3. Các bước lặp tiếp theo ---
        print("\n### 3. Các bước lặp tiếp theo")
        print("Sử dụng công thức lặp Newton cải tiến: $X_{k+1} = X_k - J(X_0)^{-1} F(X_k)$")
        
        history = [{'k': 1, 'X': X1.flatten()}]
        X_curr = X1
        
        for k in range(1, n_iterations):
            F_at_Xk = f_func(X_curr.flatten()).astype(float)
            X_next = X_curr - (J_inv_fixed @ F_at_Xk)
            
            history.append({'k': k + 1, 'X': X_next.flatten()})
            X_curr = X_next
            
            # Chỉ in chi tiết thêm bước 2 nếu có
            if k == 1:
                print(f"\n* **Tại bước $k=1$ (Tìm $X_2$):**")
                print(f"  - $F(X_1) = {matrix_to_latex(F_at_Xk, precision)}^T$")
                print(f"  - $X_2 = {matrix_to_latex(X_next, precision)}^T$")

        # --- 4. Bảng tổng hợp ---
        print("\n### 4. Bảng tổng hợp các giá trị xấp xỉ")
        header = "| k | " + " | ".join([f"$x_{i+1}$" for i in range(len(vars))]) + " |"
        sep = "| :--- | " + " | ".join([":---" for _ in range(len(vars))]) + " |"
        print(header)
        print(sep)
        print(f"| 0 | " + " | ".join([f"{v:.{precision}f}" for v in X0.flatten()]) + " |")
        
        for h in history:
            vals = " | ".join([f"{v:.{precision}f}" for v in h['X']])
            print(f"| {h['k']} | {vals} |")

        print(f"\n=> **KẾT LUẬN:** Nghiệm xấp xỉ của hệ sau {n_iterations} bước lặp là:")
        print(f"  $$X \\approx {matrix_to_latex(X_curr, precision)}$$")

    except Exception as e:
        print(f"\n[!] CÓ LỖI XẢY RA: {e}")

if __name__ == "__main__":
    newton_modified_detailed()
    input("\nNhấn Enter để thoát...")

=== CHƯƠNG TRÌNH GIẢI HỆ PHƯƠNG TRÌNH PHI TUYẾN (NEWTON CẢI TIẾN) ===


### 1. Thông tin hệ phương trình và Thiết lập ban đầu
* **Hệ phương trình:** $F(X) = \left[\begin{matrix}x^{2} + 2 y^{2} - 8 & x y + 5 y^{2} - 4\end{matrix}\right]^T$
* **Ma trận Jacobi tổng quát:** $J(X) = \left[\begin{matrix}2 x & 4 y\\y & x + 10 y\end{matrix}\right]$
* **Giá trị khởi đầu:** $X_0 = \begin{pmatrix} 2.3000000 \\ -1.1000000 \end{pmatrix}$

### 2. Chi tiết bước lặp 1 (k = 0)
* **Giá trị hàm số tại $X_0$:**
  $$F(X_0) = \begin{pmatrix} -0.2900000 \\ -0.4800000 \end{pmatrix}$$
* **Ma trận Jacobi tại $X_0$:**
  $$J(X_0) = \begin{pmatrix} 4.6000000 & -4.4000000 \\ -1.1000000 & -8.7000000 \end{pmatrix}$$
* **Ma trận nghịch đảo Jacobi (cố định cho mọi bước):**
  $$J(X_0)^{-1} = \begin{pmatrix} 0.1939367 & -0.0980829 \\ -0.0245207 & -0.1025412 \end{pmatrix}$$
* **Tính toán nghiệm xấp xỉ $X_1$:**
  $$X_1 = X_0 - J(X_0)^{-1} \cdot F(X_0)$$
  $$X_1 = \begin{pmatrix} 2.3000000 \\ -1.1000000 \end{pmatrix} - \beg